# 01 — API sample exploration

Explores dataset `jbjy-vk9h` (SECOP II — Contratos Electrónicos) before designing the extraction and the dimensional model.

Pandas is used here **only for exploration**: this notebook does not transform or load anything. All cleaning happens later in SQL.

**Goals**

1. Column inventory from the API metadata (official types and descriptions).
2. Profile of a sample: nulls, placeholder values, cardinality, examples.
3. Locate the source fields the model needs (entity, supplier, modality, signature date, value, location).
4. Check the formats the SQL cleaning must handle (values, dates, NITs, duplicates).
5. Count the candidate scopes to decide the extraction scope. Two scopes were counted: Antioquia 2022–2025 and the whole country 2022–2025. The whole country was chosen (section 7).

**Requirements:** the `secop` Conda environment and a `.env` file in the project root (README, reproduction step 2).

## 0. Setup

In [1]:
import os

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from sodapy import Socrata

# Looks for .env in the current directory and its parents (the notebook runs from notebooks/)
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError('No .env found in this directory or its parents (README, step 2).')
load_dotenv(env_path, override=True)

DOMAIN = os.environ['SOCRATA_DOMAIN']
DATASET = os.environ['SOCRATA_DATASET_ID']
client = Socrata(DOMAIN, os.environ['SOCRATA_APP_TOKEN'], timeout=120)

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 80)


def soql(**params) -> pd.DataFrame:
    """Runs a SoQL query on the server and returns the (small) result as a DataFrame."""
    return pd.DataFrame.from_records(client.get(DATASET, **params))


total = int(soql(select='count(*) AS n')['n'].iloc[0])
print(f'Dataset {DATASET} on {DOMAIN}: {total:,} rows')

Dataset jbjy-vk9h on www.datos.gov.co: 5,687,362 rows


### 0.1 Tests

In [2]:
soql(select='count(*) AS n')            # Total rows

,n
0,5687362


In [3]:
soql(where="fecha_de_firma > '2024-01-01'", limit=100) # Contracts signed since 2024

,nombre_entidad,nit_entidad,departamento,ciudad,localizaci_n,orden,sector,rama,entidad_centralizada,proceso_de_compra,...,direcci_n_de_ejecuci_n_del_contrato,criterios_de_sostenibilidad_ambiental,requisitos_ambientales_en_las_especificaciones_t_cnicas,criterios_ambientales_en_la_evaluaci_n_de_las_ofertas,obligaciones_contractuales_o_poscontractuales_a_nivel_ambiental,uso_de_etiquetado_ecol_gico,obligaciones_posconsumo,fecha_inicio_liquidacion,fecha_fin_liquidacion,ultima_actualizacion
0,CENTRO DE DIAGNOSTICO AUTOMOTOR DE POPAYAN LTDA,800253040,Cauca,Popayán,"Colombia, Cauca , Popayán",Corporación Autónoma,Transporte,Corporación Autónoma,Centralizada,CO1.BDOS.5359886,...,cra 2 No 21 dn 170\nPopayán\nCauca\nCOLOMBIA,No definido,No definido,No definido,No definido,No definido,No,NaN,NaN,NaN
1,HOSPITAL DEPARTAMENTAL UNIVERSITARIO SANTA SOFIA DE CALDAS,890801099,Caldas,Manizales,"Colombia, Caldas , Manizales",Territorial,Salud y Protección Social,Ejecutivo,Centralizada,CO1.BDOS.5356513,...,CALLE 5 # 40-02\nManizales\nCaldas\nCOLOMBIA,No definido,No definido,No definido,No definido,No definido,No,2025-01-01T00:00:00.000,2025-04-30T23:59:00.000,NaN
2,HOSPITAL DEPARTAMENTAL UNIVERSITARIO SANTA SOFIA DE CALDAS,890801099,Caldas,Manizales,"Colombia, Caldas , Manizales",Territorial,Salud y Protección Social,Ejecutivo,Centralizada,CO1.BDOS.5352160,...,CALLE 5 # 40-02\nManizales\nCaldas\nCOLOMBIA,No definido,No definido,No definido,No definido,No definido,No,2024-03-01T00:00:00.000,2024-06-29T00:00:00.000,2024-02-29T14:31:25.745
3,HOSPITAL DEPARTAMENTAL UNIVERSITARIO SANTA SOFIA DE CALDAS,890801099,Caldas,Manizales,"Colombia, Caldas , Manizales",Territorial,Salud y Protección Social,Ejecutivo,Centralizada,CO1.BDOS.5354459,...,CALLE 5 # 40-02\nManizales\nCaldas\nCOLOMBIA,No definido,No definido,No definido,No definido,No definido,No,2025-01-01T12:00:00.000,2025-04-30T11:59:00.000,NaN
4,HOSPITAL DEPARTAMENTAL UNIVERSITARIO SANTA SOFIA DE CALDAS,890801099,Caldas,Manizales,"Colombia, Caldas , Manizales",Territorial,Salud y Protección Social,Ejecutivo,Centralizada,CO1.BDOS.5349839,...,CALLE 5 # 40-02\nManizales\nCaldas\nCOLOMBIA,No definido,No definido,No definido,No definido,No definido,No,2025-01-01T12:00:00.000,2025-04-30T11:59:00.000,NaN
5,HOSPITAL DEPARTAMENTAL UNIVERSITARIO SANTA SOFIA DE CALDAS,890801099,Caldas,Manizales,"Colombia, Caldas , Manizales",Territorial,Salud y Protección Social,Ejecutivo,Centralizada,CO1.BDOS.5350223,...,CALLE 5 # 40-02\nManizales\nCaldas\nCOLOMBIA,No definido,No definido,No definido,No definido,No definido,No,2025-01-01T00:00:00.000,2025-04-30T00:00:00.000,2026-07-15T18:01:16.350
6,HOSPITAL DEPARTAMENTAL UNIVERSITARIO SANTA SOFIA DE CALDAS,890801099,Caldas,Manizales,"Colombia, Caldas , Manizales",Territorial,Salud y Protección Social,Ejecutivo,Centralizada,CO1.BDOS.5354586,...,CALLE 5 # 40-02\nManizales\nCaldas\nCOLOMBIA,No definido,No definido,No definido,No definido,No definido,No,2025-01-01T12:00:00.000,2025-03-20T11:59:00.000,2025-10-30T14:55:21.816
7,HOSPITAL DEPARTAMENTAL UNIVERSITARIO SANTA SOFIA DE CALDAS,890801099,Caldas,Manizales,"Colombia, Caldas , Manizales",Territorial,Salud y Protección Social,Ejecutivo,Centralizada,CO1.BDOS.5354194,...,CALLE 5 # 40-02\nManizales\nCaldas\nCOLOMBIA,No definido,No definido,No definido,No definido,No definido,No,2025-01-01T12:00:00.000,2025-02-07T11:59:00.000,2025-10-30T14:34:58.301
8,HOSPITAL DEPARTAMENTAL UNIVERSITARIO SANTA SOFIA DE CALDAS,890801099,Caldas,Manizales,"Colombia, Caldas , Manizales",Territorial,Salud y Protección Social,Ejecutivo,Centralizada,CO1.BDOS.5350207,...,CALLE 5 # 40-02\nManizales\nCaldas\nCOLOMBIA,No definido,No definido,No definido,No definido,No definido,No,2025-01-01T00:00:00.000,2025-04-30T00:00:00.000,2024-12-06T15:58:44.702
9,EMPRESA SOCIAL DEL ESTADO E.S.E HOSPITAL LA MILAGROSA,890702476,Tolima,Villarrica,"Colombia, Tolima , Villarrica",Territorial,Salud y Protección Social,Ejecutivo,Centralizada,CO1.BDOS.5360601,...,Cra 4 Calle 1 #

## 1. Column inventory (API metadata)

The API returns every value as text in JSON. The metadata gives the official type of each column, which tells us what conversion each one will need in SQL.

In [4]:
meta = client.get_metadata(DATASET)
columns = pd.DataFrame(
    [
        {
            'field': c.get('fieldName'),
            'name': c.get('name'),
            'api_type': c.get('dataTypeName'),
            'description': (c.get('description') or '').strip()[:120],
        }
        for c in meta['columns']
        if not c.get('fieldName', '').startswith(':')  # skip system columns
    ]
)
types = dict(zip(columns['field'], columns['api_type']))

updated = pd.to_datetime(meta.get('rowsUpdatedAt'), unit='s')
print(f'{len(columns)} columns; data last updated: {updated}')
columns['api_type'].value_counts()

95 columns; data last updated: 2026-11-07 13:20:06


api_type
text             64
number           19
calendar_date    11
url               1
Name: count, dtype: int64

In [5]:
columns

,field,name,api_type,description
0,nombre_entidad,Nombre Entidad,text,Nombre de la entidad del estado que publica el contrato
1,nit_entidad,Nit Entidad,number,NIT de la entidad del estado que publica el contrato
2,departamento,Departamento,text,Departamento en el cual se registró la entidad del estado que publica el con...
3,ciudad,Ciudad,text,Ciudad en el cual se registró la entidad del estado que publica el contrato
4,localizaci_n,Localización,text,Ubicación completa de la entidad del estado que publica el contrato
5,orden,Orden,text,Orden entidad del estado que publica el contrato
6,sector,Sector,text,Sector entidad del estado que publica el contrato
7,rama,Rama,text,Rama del estado de la entidad que publica el contrato
8,entidad_centralizada,Entidad Centralizada,text,Define si la entidad es descentralizada o centralizada
9,proceso_de_compra,Proceso de Compra,text,Identificador del proceso de compra publicado


### 1.2 Column inventory (SoQL query):

In [6]:
df = soql(limit=100)
schema = (
    pd.DataFrame({
        "columna": df.columns,
        "tipo":   [str(t) for t in df.dtypes],
    })
    .sort_values("columna")
    .reset_index(drop=True)
)
schema

,columna,tipo
0,ciudad,str
1,codigo_de_categoria_principal,str
2,codigo_entidad,str
3,codigo_proveedor,str
4,condiciones_de_entrega,str
5,criterios_ambientales_en_la_evaluaci_n_de_las_ofertas,str
6,criterios_de_sostenibilidad_ambiental,str
7,departamento,str
8,descripcion_del_proceso,str
9,descripcion_documentos_tipo,str


## 2. Sample

5,000 rows ordered by the internal `:id`. The order is stable, so the sample is reproducible, but it is **not random**: treat proportions from it as indicative only. The full-dataset counts in section 6 come from the server.

The API omits fields that are null in a row, so the sample is reindexed to the full column list.

In [7]:
SAMPLE_SIZE = 5_000
sample = soql(limit=SAMPLE_SIZE, order=':id').reindex(columns=columns['field'])
print(sample.shape)
sample.head(4)

(5000, 95)


field,nombre_entidad,nit_entidad,departamento,ciudad,localizaci_n,orden,sector,rama,entidad_centralizada,proceso_de_compra,...,criterios_de_sostenibilidad_ambiental,requisitos_ambientales_en_las_especificaciones_t_cnicas,criterios_ambientales_en_la_evaluaci_n_de_las_ofertas,obligaciones_contractuales_o_poscontractuales_a_nivel_ambiental,uso_de_etiquetado_ecol_gico,obligaciones_posconsumo,fecha_inicio_obligaciones_posconsumo,fecha_fin_obligaciones_posconsumo,fecha_inicio_reversi_n,fecha_fin_reversi_n
0,DANE - DIRECCION TERRITORIAL NORTE,899999027,Atlántico,No Definido,"Colombia, Atlántico , No Definido",Nacional,Información Estadística,Ejecutivo,Centralizada,CO1.BDOS.6658824,...,No definido,No definido,No definido,No definido,No definido,No,NaN,NaN,NaN,NaN
1,GOBERNACION DEL QUINDIO ARMENIA,890001639,Quindío,Armenia,"Colombia, Quindío, Armenia",Territorial,No aplica/No pertenece,Ejecutivo,Centralizada,CO1.BDOS.7744023,...,No definido,No definido,No definido,No definido,No definido,No,NaN,NaN,NaN,NaN
2,IGAC Sede Central *,899999004,Distrito Capital de Bogotá,No Definido,"Colombia, Bogotá, No Definido",Nacional,Información Estadística,Ejecutivo,Centralizada,CO1.BDOS.9606774,...,No definido,No definido,No definido,No definido,No definido,No,NaN,NaN,NaN,NaN
3,CORPORACION AUTONOMA REGIONAL DE LA GUAJIRA,892115314,La Guajira,Riohacha,"Colombia, La Guajira , Riohacha",Corporación Autónoma,Ambiente y Desarrollo Sostenible,Corporación Autónoma,Centralizada,CO1.BDOS.7704855,...,No definido,No definido,No definido,No definido,No definido,No,NaN,NaN,NaN,NaN


## 3. Column profile

Open-data sources often fill missing fields with text such as `No Definido` or `No Aplica`. These are nulls in disguise, so the profile counts them separately (`placeholder_pct`). Extend `PLACEHOLDERS` if the examples show others.

In [8]:
PLACEHOLDERS = {'', '-', 'n/a', 'na', 'null', 'none', 'no definido', 'no aplica',
                'sin descripcion', 'sin descripción'}


def profile(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    rows = []
    for col in df.columns:
        s = df[col]
        text = s.dropna().astype(str).str.strip()
        is_placeholder = text.str.lower().isin(PLACEHOLDERS)
        real = text[~is_placeholder]
        rows.append({
            'field': col,
            'api_type': types.get(col),
            'null_pct': round(100 * s.isna().mean(), 1),
            'placeholder_pct': round(100 * is_placeholder.sum() / n, 1),
            'distinct': real.nunique(),
            'examples': ' | '.join(real.drop_duplicates().head(3).str[:40]),
        })
    return pd.DataFrame(rows).set_index('field')


prof = profile(sample)
prof

,api_type,null_pct,placeholder_pct,distinct,examples
field,,,,,
nombre_entidad,text,0.0,0.0,1262,DANE - DIRECCION TERRITORIAL NORTE | GOBERNACION DEL QUINDIO ARMENIA | IGAC ...
nit_entidad,number,0.0,0.0,1035,899999027 | 890001639 | 899999004
departamento,text,0.0,2.0,32,Atlántico | Quindío | Distrito Capital de Bogotá
ciudad,text,0.0,21.8,354,Armenia | Riohacha | Duitama
localizaci_n,text,0.0,0.0,426,"Colombia, Atlántico , No Definido | Colombia, Quindío, Armenia | Colombia, ..."
orden,text,0.0,0.0,3,Nacional | Territorial | Corporación Autónoma
sector,text,0.0,0.0,24,Información Estadística | No aplica/No pertenece | Ambiente y Desarrollo Sos...
rama,text,0.0,0.0,4,Ejecutivo | Corporación Autónoma | Legislativo
entidad_centralizada,text,0.0,0.0,2,Centralizada | Descentralizada


In [9]:
# Columns that are almost always empty in the sample: candidates to leave out of the model
prof[prof['null_pct'] + prof['placeholder_pct'] >= 90]

,api_type,null_pct,placeholder_pct,distinct,examples
field,,,,,
puntos_del_acuerdo,text,0.0,99.5,3,ReformaRuralIntegral | FinConflicto | VictimasConflicto
pilares_del_acuerdo,text,0.0,99.5,4,OSPRUDS | RFARCVC | JusticiaYVerdad
fecha_de_notificaci_n_de_prorrogaci_n,calendar_date,90.3,0.0,406,2023-12-16T04:59:00.000 | 2024-02-01T04:00:00.000 | 2026-12-01T04:59:00.000
descripcion_documentos_tipo,text,0.0,99.6,6,Catastro multipropósito - Contratos inte | Infraestructura de transporte | C...
criterios_de_sostenibilidad_ambiental,text,0.0,98.6,2,No | Sí
requisitos_ambientales_en_las_especificaciones_t_cnicas,text,0.0,99.9,2,Sí | No
criterios_ambientales_en_la_evaluaci_n_de_las_ofertas,text,0.0,99.9,2,Sí | No
obligaciones_contractuales_o_poscontractuales_a_nivel_ambiental,text,0.0,99.9,2,Sí | No
uso_de_etiquetado_ecol_gico,text,0.0,99.9,1,No


## 4. Source fields for the model

`CANDIDATES` maps each role in the dimensional model to the source field expected to fill it. The table checks whether each one exists and shows its profile. If a field is missing or looks wrong, use `find()` to search the inventory and **correct the mapping here**. The confirmed names are what the staging table and the model will use.

In [10]:
CANDIDATES = {
    'contract id': 'id_contrato',
    'entity NIT': 'nit_entidad',
    'entity name': 'nombre_entidad',
    'entity level': 'orden',
    'entity sector': 'sector',
    'supplier document type': 'tipodocproveedor',
    'supplier document': 'documento_proveedor',
    'supplier name': 'proveedor_adjudicado',
    'selection modality': 'modalidad_de_contratacion',
    'contract type': 'tipo_de_contrato',
    'contract status': 'estado_contrato',
    'signature date': 'fecha_de_firma',
    'start date': 'fecha_de_inicio_del_contrato',
    'end date': 'fecha_de_fin_del_contrato',
    'contract value': 'valor_del_contrato',
    'department': 'departamento',
    'city': 'ciudad',
    'last update': 'ultima_actualizacion',
}

mapping = pd.DataFrame(
    [{'role': role, 'field': f, 'exists': f in types} for role, f in CANDIDATES.items()]
).set_index('role')
display(mapping.join(prof, on='field'))

missing = mapping.loc[~mapping['exists'], 'field'].tolist()
print('All candidate fields exist.' if not missing else f'Not found, search with find(): {missing}')

,field,exists,api_type,null_pct,placeholder_pct,distinct,examples
role,,,,,,,
contract id,id_contrato,True,text,0.0,0.0,5000,CO1.PCCNTR.6730680 | CO1.PCCNTR.7594386 | CO1.PCCNTR.8988493
entity NIT,nit_entidad,True,number,0.0,0.0,1035,899999027 | 890001639 | 899999004
entity name,nombre_entidad,True,text,0.0,0.0,1262,DANE - DIRECCION TERRITORIAL NORTE | GOBERNACION DEL QUINDIO ARMENIA | IGAC ...
entity level,orden,True,text,0.0,0.0,3,Nacional | Territorial | Corporación Autónoma
entity sector,sector,True,text,0.0,0.0,24,Información Estadística | No aplica/No pertenece | Ambiente y Desarrollo Sos...
supplier document type,tipodocproveedor,True,text,0.0,0.3,5,Cédula de Ciudadanía | NIT | Otro
supplier document,documento_proveedor,True,text,0.0,0.4,4936,1042440422 | 1094728647 | 1061701920
supplier name,proveedor_adjudicado,True,text,0.0,0.0,4955,MAROLIN MARTINEZ HERRERA | MARIA ALEJANDRA GIRALDO CRUZ | EDDY ALEJANDRA CAM...
selection modality,modalidad_de_contratacion,True,text,0.0,0.0,11,Contratación directa | Contratación régimen especial | Mínima cuantía


All candidate fields exist.


In [11]:
def find(*words: str) -> pd.DataFrame:
    """Columns whose field name or display name contains any of the words."""
    pattern = '|'.join(words)
    hit = (columns['field'].str.contains(pattern, case=False)
           | columns['name'].str.contains(pattern, case=False))
    return columns.loc[hit, ['field', 'name', 'api_type']].join(
        prof[['null_pct', 'examples']], on='field')


for words in [('depart', 'munic', 'ciudad', 'dane', 'localiz'),
              ('valor',),
              ('fecha',),
              ('proveedor', 'documento'),
              ('adicion', 'dias', 'duraci')]:
    print('Search:', ', '.join(words))
    display(find(*words))

Search: depart, munic, ciudad, dane, localiz


,field,name,api_type,null_pct,examples
2,departamento,Departamento,text,0.0,Atlántico | Quindío | Distrito Capital de Bogotá
3,ciudad,Ciudad,text,0.0,Armenia | Riohacha | Duitama
4,localizaci_n,Localización,text,0.0,"Colombia, Atlántico , No Definido | Colombia, Quindío, Armenia | Colombia, ..."


Search: valor


,field,name,api_type,null_pct,examples
34,valor_del_contrato,Valor del Contrato,number,0.0,10007333 | 3700000 | 35700000
35,valor_de_pago_adelantado,Valor de pago adelantado,number,0.0,0 | 11400000 | 39750847.6
36,valor_facturado,Valor Facturado,number,0.0,9757333 | 3700000 | 34720000
37,valor_pendiente_de_pago,Valor Pendiente de Pago,number,0.0,250000 | 3700000 | 5180000
38,valor_pagado,Valor Pagado,number,0.0,9757333 | 0 | 30520000
39,valor_amortizado,Valor Amortizado,number,0.0,0 | 360234746.74
40,valor_pendiente_de,Valor Pendiente de Amortizacion,number,0.0,0 | 11400000 | 39750847.6
41,valor_pendiente_de_ejecucion,Valor Pendiente de Ejecucion,number,0.0,250000 | 3700000 | 5180000


Search: fecha


,field,name,api_type,null_pct,examples
18,fecha_de_firma,Fecha de Firma,calendar_date,1.1,2024-09-06T14:36:22.403 | 2025-03-03T19:13:06.908 | 2026-01-20T15:02:43.134
19,fecha_de_inicio_del_contrato,Fecha de Inicio del Contrato,calendar_date,1.8,2024-09-07T12:00:00.000 | 2025-03-03T19:15:00.000 | 2026-01-23T08:00:00.000
20,fecha_de_fin_del_contrato,Fecha de Fin del Contrato,calendar_date,0.0,2024-12-31T18:00:00.000 | 2025-04-01T23:59:00.000 | 2026-10-07T10:00:00.000
64,fecha_inicio_liquidacion,Fecha Inicio Liquidacion,calendar_date,89.0,2026-12-16T08:00:00.000 | 2017-12-22T06:00:00.000 | 2030-01-01T23:59:00.000
65,fecha_fin_liquidacion,Fecha Fin Liquidacion,calendar_date,89.0,2027-04-15T18:00:00.000 | 2018-04-12T18:00:00.000 | 2032-06-01T23:59:00.000
72,fecha_de_notificaci_n_de_prorrogaci_n,Fecha de notificación de prorrogación,calendar_date,90.3,2023-12-16T04:59:00.000 | 2024-02-01T04:00:00.000 | 2026-12-01T04:59:00.000
91,fecha_inicio_obligaciones_posconsumo,Fecha inicio obligaciones posconsumo,calendar_date,100.0,2023-11-06T05:00:00.000 | 2021-08-22T13:00:00.000
92,fecha_fin_obligaciones_posconsumo,Fecha fin obligaciones posconsumo,calendar_date,100.0,2026-11-06T04:59:00.000 | 2022-08-22T22:00:00.000
93,fecha_inicio_reversi_n,Fecha inicio reversión,calendar_date,100.0,2018-12-29T06:00:00.000 | 2024-12-31T17:00:00.000
94,fecha_fin_reversi_n,Fecha fin reversión,calendar_date,100.0,2019-01-29T04:59:00.000 | 2025-04-30T17:00:00.000


Search: proveedor, documento


,field,name,api_type,null_pct,examples
22,tipodocproveedor,TipoDocProveedor,text,0.0,Cédula de Ciudadanía | NIT | Otro
23,documento_proveedor,Documento Proveedor,text,0.0,1042440422 | 1094728647 | 1061701920
24,proveedor_adjudicado,Proveedor Adjudicado,text,0.0,MAROLIN MARTINEZ HERRERA | MARIA ALEJANDRA GIRALDO CRUZ | EDDY ALEJANDRA CAM...
63,codigo_proveedor,Codigo Proveedor,text,0.0,728258468 | 725637540 | 718801111
74,tipo_de_documento_ordenador_del_gasto,Tipo de documento Ordenador del gasto,text,0.0,Cédula de Ciudadanía | Otro | Cédula de Extranjería
75,n_mero_de_documento_ordenador_del_gasto,Número de documento Ordenador del gasto,text,0.0,79540901 | 4470927 | 34323671
77,tipo_de_documento_supervisor,Tipo de documento supervisor,text,0.0,Cédula de Ciudadanía | Citizenship Identification | Otro
78,n_mero_de_documento_supervisor,Número de documento supervisor,text,0.0,1043612774 | 10265612 | 1079390785
80,tipo_de_documento_ordenador_de_pago,Tipo de documento Ordenador de Pago,text,0.0,Cédula de Ciudadanía | Otro | Tarjeta de Identidad
81,n_mero_de_documento_ordenador_de_pago,Número de documento Ordenador de Pago,text,0.0,34323671 | 63536268. | 60349374


Search: adicion, dias, duraci


,field,name,api_type,null_pct,examples
45,dias_adicionados,Dias adicionados,number,0.0,0 | 110 | 153
67,duraci_n_del_contrato,Duración del contrato,text,0.0,118 Dia(s) | 30 Dia(s) | 255 Dia(s)


## 5. Formats the SQL cleaning must handle

Conversions here use `errors='coerce'` only to **measure** how many values would fail; the real conversion is done later in SQL.

In [12]:
# Monetary value
VALUE = CANDIDATES['contract value']
v = sample[VALUE].dropna().astype(str).str.strip()
num = pd.to_numeric(v, errors='coerce')
print(f'{VALUE}: {len(v):,} non-null | {num.isna().sum():,} not numeric | '
      f'{(num == 0).sum():,} zero | {(num < 0).sum():,} negative')
print('Non-numeric examples:', v[num.isna()].unique()[:5])
num.describe(percentiles=[0.5, 0.9, 0.99]).map(lambda x: f'{x:,.0f}')

valor_del_contrato: 5,000 non-null | 0 not numeric | 43 zero | 0 negative
Non-numeric examples: <ArrowStringArray>
[]
Length: 0, dtype: str


count              5,000
mean         138,314,189
std        2,204,900,012
min                    0
50%           20,000,000
90%           85,800,000
99%        1,494,532,160
max      133,242,583,045
Name: valor_del_contrato, dtype: str

In [13]:
# Dates
for role in ['signature date', 'start date', 'end date', 'last update']:
    col = CANDIDATES[role]
    s = sample[col].dropna().astype(str).str.strip()
    if s.empty:
        print(f'{col}: empty in the sample')
        continue
    d = pd.to_datetime(s, errors='coerce', format='ISO8601')
    print(f'{col}: {len(s):,} non-null | {d.isna().sum():,} unparseable | '
          f'range {d.min()} -> {d.max()} | raw example {s.iloc[0]!r}')

fecha_de_firma: 4,947 non-null | 0 unparseable | range 2017-02-02 14:35:51.961000 -> 2026-11-05 10:41:46.572000 | raw example '2024-09-06T14:36:22.403'
fecha_de_inicio_del_contrato: 4,909 non-null | 0 unparseable | range 2017-01-31 00:00:00 -> 2026-11-04 12:00:00 | raw example '2024-09-07T12:00:00.000'
fecha_de_fin_del_contrato: 5,000 non-null | 0 unparseable | range 2017-05-30 16:00:00 -> 2032-01-18 12:00:00 | raw example '2024-12-31T18:00:00.000'
ultima_actualizacion: 3,173 non-null | 0 unparseable | range 2018-01-17 14:28:45.605000 -> 2026-11-05 08:42:16.160000 | raw example '2026-03-17T13:54:58.326'


In [14]:
# Document (NIT) formats: digits become 9 and letters become A, so each distinct shape is one format
def shape(s: pd.Series) -> pd.Series:
    return (s.str.replace(r'\d', '9', regex=True)
             .str.replace(r'[^\W\d_]', 'A', regex=True))


display(sample[CANDIDATES['supplier document type']].value_counts(dropna=False).head(10))
for role in ['supplier document', 'entity NIT']:
    col = CANDIDATES[role]
    s = sample[col].dropna().astype(str).str.strip()
    print(f'{col}: most common formats')
    display(shape(s).value_counts().head(10))

tipodocproveedor
Cédula de Ciudadanía     4236
NIT                       728
No Definido                16
Otro                       11
Cédula de Extranjería       7
Tarjeta de Identidad        2
Name: count, dtype: int64

documento_proveedor: most common formats


documento_proveedor
9999999999             2248
99999999               1867
999999999               671
9999999                 177
AA AAAAAAAA              19
999999                    3
9.999.999.999             3
99999999.                 2
99999999999               2
9 9 9 9 9 9 9 9 9 9       1
Name: count, dtype: int64

nit_entidad: most common formats


nit_entidad
999999999     4590
9999999999     408
99999999         2
Name: count, dtype: int64

In [15]:
# Duplicates of the contract key (versions of the same contract)
ID = CANDIDATES['contract id']
dup = sample[ID].duplicated(keep=False)
print(f'{ID}: {sample[ID].nunique():,} distinct in {len(sample):,} rows | {dup.sum():,} rows share an id')
show = [ID, CANDIDATES['contract status'], CANDIDATES['contract value'], CANDIDATES['last update']]
sample.loc[dup, show].sort_values(ID).head(10)

id_contrato: 5,000 distinct in 5,000 rows | 0 rows share an id


field,id_contrato,estado_contrato,valor_del_contrato,ultima_actualizacion


In [16]:
# Categorical fields: values and frequencies in the sample
for role in ['selection modality', 'contract type', 'contract status', 'entity level']:
    col = CANDIDATES[role]
    print(col)
    display(sample[col].value_counts(dropna=False).head(20))

modalidad_de_contratacion


modalidad_de_contratacion
Contratación directa                           3822
Contratación régimen especial                   690
Mínima cuantía                                  266
Contratación Directa (con ofertas)               63
Contratación régimen especial (con ofertas)      51
Selección Abreviada de Menor Cuantía             38
Selección abreviada subasta inversa              32
Licitación pública                               16
Concurso de méritos abierto                      13
Licitación pública Obra Publica                   8
Licitación Pública Acuerdo Marco de Precios       1
Name: count, dtype: int64

tipo_de_contrato


tipo_de_contrato
Prestación de servicios           4327
Otro                               180
Decreto 092 de 2017                158
Suministros                        135
Compraventa                         71
Obra                                44
Arrendamiento de inmuebles          43
Interventoría                       14
Comodato                            12
Consultoría                          7
Seguros                              3
Arrendamiento de muebles             2
Acuerdo Marco de Precios             2
Operaciones de Crédito Público       1
No Especificado                      1
Name: count, dtype: int64

estado_contrato


estado_contrato
Cerrado              1546
En ejecución         1517
Modificado           1006
terminado             690
Aprobado              167
enviado Proveedor      28
En aprobación          25
cedido                 16
Suspendido              5
Name: count, dtype: int64

orden


orden
Territorial             3260
Nacional                1673
Corporación Autónoma      67
Name: count, dtype: int64

## 6. Scope count (server-side SoQL)

These queries aggregate on the server over the whole dataset; only the small results are downloaded. `date_extract_y` requires the signature date to be a `calendar_date` column (check `api_type` in section 1).

In [17]:
DEPT = CANDIDATES['department']
DATE = CANDIDATES['signature date']

by_dept = soql(select=f'{DEPT}, count(*) AS n', group=DEPT, order='count(*) DESC', limit=200)
by_dept['n'] = by_dept['n'].astype(int)
by_dept['pct'] = (100 * by_dept['n'] / total).round(2)
by_dept

,departamento,n,pct
0,Distrito Capital de Bogotá,1873653,32.94
1,Valle del Cauca,582841,10.25
2,Antioquia,533040,9.37
3,Santander,263982,4.64
4,Cundinamarca,247074,4.34
5,Atlántico,189891,3.34
6,Bolívar,168260,2.96
7,Tolima,158072,2.78
8,Boyacá,144265,2.54
9,Norte de Santander,140822,2.48


In [18]:
# The exact spelling of the department as stored in the source (it may differ in case or accents)
matches = by_dept[by_dept[DEPT].str.contains('antioquia', case=False, na=False)]
display(matches)
SCOPE_DEPT = matches[DEPT].iloc[0]
print(f'Department value used for the scope: {SCOPE_DEPT!r}')

,departamento,n,pct
2,Antioquia,533040,9.37


Department value used for the scope: 'Antioquia'


In [19]:
# Contracts per signature year in the department (a null year = no signature date)
year = f'date_extract_y({DATE})'
by_year = soql(select=f'{year} AS anio, count(*) AS n', where=f"{DEPT} = '{SCOPE_DEPT}'",
               group=year, order=year, limit=200)
by_year['n'] = by_year['n'].astype(int)
by_year

,anio,n
0,2017,365
1,2018,8155
2,2019,6414
3,2020,13349
4,2021,47228
5,2022,70512
6,2023,85969
7,2024,85569
8,2025,107056
9,2026,102594


In [20]:
START, END = '2022-01-01T00:00:00', '2026-01-01T00:00:00'  # END is exclusive
scope_where = f"{DEPT} = '{SCOPE_DEPT}' AND {DATE} >= '{START}' AND {DATE} < '{END}'"
scope_n = int(soql(select='count(*) AS n', where=scope_where)['n'].iloc[0])
no_date = int(soql(select='count(*) AS n', where=f"{DEPT} = '{SCOPE_DEPT}' AND {DATE} IS NULL")['n'].iloc[0])

PAGE = 50_000
bytes_per_row = sample.memory_usage(deep=True).sum() / len(sample)
print(f'Proposed scope ({SCOPE_DEPT}, signed 2022-2025): {scope_n:,} rows '
      f'({100 * scope_n / total:.1f}% of the dataset)')
print(f'Rows in {SCOPE_DEPT} without signature date (outside any date range): {no_date:,}')
print(f'Extraction: about {-(-scope_n // PAGE)} pages of {PAGE:,} rows; '
      f'roughly {scope_n * bytes_per_row / 1e9:.1f} GB as an in-memory DataFrame (Parquet on disk is much smaller)')

Proposed scope (Antioquia, signed 2022-2025): 349,106 rows (6.1% of the dataset)
Rows in Antioquia without signature date (outside any date range): 5,829
Extraction: about 7 pages of 50,000 rows; roughly 0.8 GB as an in-memory DataFrame (Parquet on disk is much smaller)


In [21]:
# Server-side duplicate check over the whole proposed scope (the sample in section 5 had none).
# Full SoQL through $query because sodapy has no keyword argument for HAVING.
dups = soql(query=(
    f"SELECT {ID}, count(*) AS n WHERE {scope_where} "
    f"GROUP BY {ID} HAVING count(*) > 1 ORDER BY count(*) DESC LIMIT 20"
))
print(f'{ID} values repeated in the scope (up to 20 shown): {len(dups)}')
dups

id_contrato values repeated in the scope (up to 20 shown): 0


""


In [22]:
# Nationwide alternative: contracts per signature year in the whole country, and duplicates nationwide
by_year_all = soql(select=f'{year} AS anio, count(*) AS n', group=year, order=year, limit=200)
by_year_all['n'] = by_year_all['n'].astype(int)
display(by_year_all)

nat_where = f"{DATE} >= '{START}' AND {DATE} < '{END}'"
nat_n = int(soql(select='count(*) AS n', where=nat_where)['n'].iloc[0])
print(f'Whole country, signed 2022-2025: {nat_n:,} rows | about {-(-nat_n // PAGE)} pages of {PAGE:,}')

dups_all = soql(query=(
    f"SELECT {ID}, count(*) AS n GROUP BY {ID} HAVING count(*) > 1 "
    f"ORDER BY count(*) DESC LIMIT 20"
))
print(f'{ID} values repeated in the whole dataset (up to 20 shown): {len(dups_all)}')
dups_all

,anio,n
0,2015,10
1,2016,1340
2,2017,22224
3,2018,142586
4,2019,142090
5,2020,356623
6,2021,560441
7,2022,707765
8,2023,835462
9,2024,881978


Whole country, signed 2022-2025: 3,442,281 rows | about 69 pages of 50,000
id_contrato values repeated in the whole dataset (up to 20 shown): 20


,id_contrato,n
0,CO1.PCCNTR.5621949,2
1,CO1.PCCNTR.5524450,2
2,CO1.PCCNTR.4009175,2
3,CO1.PCCNTR.8384570,2
4,CO1.PCCNTR.7966960,2
5,CO1.PCCNTR.5717141,2
6,CO1.PCCNTR.6662233,2
7,CO1.PCCNTR.4028558,2
8,CO1.PCCNTR.4722039,2
9,CO1.PCCNTR.5558943,2


In [23]:
# Duplicates in the nationwide scope (signed 2022-2025): how many ids repeat, and what differs between the copies
dup_nat = soql(query=(
    f"SELECT {ID}, count(*) AS n WHERE {nat_where} "
    f"GROUP BY {ID} HAVING count(*) > 1 ORDER BY count(*) DESC LIMIT 50000"
))
if dup_nat.empty:
    print('No repeated id_contrato in the nationwide 2022-2025 scope.')
else:
    dup_nat['n'] = dup_nat['n'].astype(int)
    note = ' (truncated at 50,000: there are more)' if len(dup_nat) == 50_000 else ''
    print(f'Repeated ids in scope: {len(dup_nat):,}{note} | rows involved: {dup_nat["n"].sum():,} '
          f'| extra copies: {(dup_nat["n"] - 1).sum():,} | max copies of one id: {dup_nat["n"].max()}')
    ids = ', '.join(f"'{i}'" for i in dup_nat[ID].head(3))
    detail = soql(
        select=(f':id, {ID}, proceso_de_compra, nombre_entidad, documento_proveedor, '
                f'proveedor_adjudicado, estado_contrato, valor_del_contrato, {DATE}, ultima_actualizacion'),
        where=f'{ID} IN ({ids})', order=f'{ID}, :id',
    )
    display(detail)

Repeated ids in scope: 254 | rows involved: 508 | extra copies: 254 | max copies of one id: 2


,:id,id_contrato,proceso_de_compra,nombre_entidad,documento_proveedor,proveedor_adjudicado,estado_contrato,valor_del_contrato,fecha_de_firma,ultima_actualizacion
0,row-nyb9-m2cc~wcrm,CO1.PCCNTR.4009175,CO1.BDOS.3257531,CONCEJO MUNICIPAL DE PIEDECUESTA,1102370553,Martha Briceño Valencia,Modificado,6933333,2022-09-09T13:45:04.986,2022-12-07T15:35:13.643
1,row-232d~g3cm-3q2y,CO1.PCCNTR.4009175,CO1.BDOS.3257531,CONCEJO MUNICIPAL DE PIEDECUESTA,1102370553,Martha Briceño Valencia,Modificado,6933333,2022-09-09T13:45:04.986,2022-12-07T15:35:13.643
2,row-dhpz_8gye~2c3f,CO1.PCCNTR.5524450,CO1.BDOS.5129114,CONCEJO MUNICIPAL DE PIEDECUESTA,1102366892,Silvia Marcela Rodriguez Diaz,Modificado,3000000,2023-11-07T22:47:33.857,2023-12-22T10:17:16.255
3,row-ew3w~v6k4.7w6t,CO1.PCCNTR.5524450,CO1.BDOS.5129114,CONCEJO MUNICIPAL DE PIEDECUESTA,1102366892,Silvia Marcela Rodriguez Diaz,Modificado,3000000,2023-11-07T22:47:33.857,2023-12-22T10:17:16.255
4,row-yuwa~3f9d_qg56,CO1.PCCNTR.5621949,CO1.BDOS.5256778,CONCEJO MUNICIPAL DE PIEDECUESTA,1095786817,ANA MILENA ACOSTA RUEDA,En ejecución,1800000,2023-12-01T23:33:14.844,NaN
5,row-s6dq_xwar_pdjy,CO1.PCCNTR.5621949,CO1.BDOS.5256778,CONCEJO MUNICIPAL DE PIEDECUESTA,1095786817,ANA MILENA ACOSTA RUEDA,En ejecución,1800000,2023-12-01T23:33:14.844,NaN


In [24]:
client.close()

## 7. Findings

Run of 2026-11-07 (data last updated 2026-11-07 13:20). Sample = first 5,000 rows by `:id`, not random: sample percentages are indicative, server counts are exact.

### Source structure
- The metadata lists **95 columns**: 64 `text`, 19 `number`, 11 `calendar_date` and 1 `url`. The API leaves null fields out of each JSON row, so a single row shows fewer keys: 87 in the token check, 91 across a 100-row query. **Staging must take the column list from the metadata, not from the first page.**
- 13 columns are ≥ 90 % empty or placeholder in the sample, including the peace-agreement fields, the environmental criteria and the post-consumption and reversion dates. They stay in staging but are left out of the model.

### Confirmed source fields (`CANDIDATES`)
All 18 expected fields exist with the expected content:

| Role | Field | API type | Note |
|---|---|---|---|
| contract id | `id_contrato` | text | `CO1.PCCNTR.<n>`; 254 ids repeat in the scope (see Duplicates) |
| entity | `nit_entidad`, `nombre_entidad`, `orden`, `sector` | number / text | `orden` = level (Territorial, Nacional, Corporación Autónoma) |
| supplier | `tipodocproveedor`, `documento_proveedor`, `proveedor_adjudicado` | text | |
| modality | `modalidad_de_contratacion` | text | 11 values in the sample |
| type, status | `tipo_de_contrato`, `estado_contrato` | text | |
| dates | `fecha_de_firma`, `fecha_de_inicio_del_contrato`, `fecha_de_fin_del_contrato`, `ultima_actualizacion` | calendar_date | |
| value | `valor_del_contrato` | number | |
| location | `departamento`, `ciudad` | text | no DANE code column exists |

Other useful fields: `dias_adicionados` (number), `duraci_n_del_contrato` (text such as `118 Dia(s)`), `codigo_entidad`, `codigo_proveedor`, and `localizaci_n` (`Colombia, <dept>, <city>`, irregular spacing; redundant).

**Effects on the model (provisional design):**
- `dim_ubicacion` cannot use `codigo_dane` from the source. The dashboard map needs DANE department codes, so they will come from a DIVIPOLA reference table joined by normalized department name.
- `departamento` appears to be the location of the contracting **entity**, not where the work is done. This is not verified yet.
- `numero_adiciones` does not exist. `dias_adicionados` is available instead.
- `nivel` comes from `orden`.
- `tipo_persona` can be derived from `tipodocproveedor`.

### Formats to clean in SQL
- **Value:** 100 % numeric, 0 negatives, 43 zeros (0.9 %). Median 20,000,000 COP, p99 ≈ 1.49 billion, max ≈ 133.2 billion. The heavy tail matters for outliers (question 7) and for the Pareto analysis.
- **Dates:** ISO 8601 with milliseconds (`2024-09-06T14:36:22.403`), none unparseable. Nulls in the sample: signature 1.1 %, start 1.8 %, last update 36.5 %. End dates reach 2032 (future dates are valid for contracts still in execution).
- **Supplier document:** mostly 7–10 digits. There are also variants with thousands dots (`9.999.999.999`), a trailing dot (`99999999.`), spaced digits (`9 9 9 9 …`) and text (`AA AAAAAAAA`, 19 rows, probably `No Definido`). Normalization: keep digits only and treat text as unknown.
- **Supplier type:** Cédula de Ciudadanía 84.7 %, NIT 14.6 %, and the rest are No Definido, Otro, CE or TI. **Most suppliers are natural persons.** This is relevant when interpreting concentration.
- **Entity NIT:** clean, 9–10 digits (`number` type).
- **Placeholders:** `ciudad` is `No Definido` in 21.8 % of the sample and `departamento` in 2.0 %. Across the whole dataset, 109,831 rows have `departamento = 'No Definido'`.
- **Categories:**
  - `estado_contrato` mixes case (`terminado`, `enviado Proveedor`, `cedido`) and needs normalization.
  - `modalidad_de_contratacion` has `(con ofertas)` variants. Whether to group them is a model decision.
  - In the sample, Contratación directa is 76 % of contracts and Prestación de servicios is 87 %.

### Duplicates
- Sample, and Antioquia 2022–2025: no repeated `id_contrato`.
- Whole country, signed 2022–2025: **254 ids appear twice** (508 rows, 254 extra copies, never more than 2 copies).
- In the 3 cases inspected, the two rows have different `:id` but are identical in every column shown, including value, status, dates and `ultima_actualizacion`. They look like publication duplicates, not contract versions.
- **Rule for SQL:** keep one row per `id_contrato`, preferring the latest `ultima_actualizacion` and then the lowest `:id`. Count and record the dropped copies (expected: 254).
- **Consequences:**
  - The extraction must download the system column `:id`; the API does not return it unless it is selected.
  - Staging must check whether the copies are identical in all 95 columns, not only the ones shown.
  - Expected contracts after deduplication: 3,442,281 − 254 = 3,442,027.

### Scope — confirmed 2026-11-07: whole country, signed 2022–2025
- Signed per year nationwide: 2022 707,765 · 2023 835,462 · 2024 881,978 · 2025 1,017,076. 2026 has 952,664 so far, but the year is incomplete.
- **Scope: 3,442,281 rows** (60.5 % of the dataset), 69 pages of 50,000.
- Left out, 2,245,081 rows in total:

  | Reason | Rows |
  |---|---|
  | Signed 2015–2021 (few contracts in the early years while SECOP II was being adopted: 10 in 2015, 1,340 in 2016) | 1,225,314 |
  | Signed in 2026 (incomplete year) | 952,664 |
  | No signature date | 67,103 |
- Reason for the change from Antioquia (349,106 rows): the dashboard will show a map of Colombia by department, and the department comparison and question 5 need every department.
- Storage: the PostgreSQL data folder is on `/`, which has 28 GB free. The project folder (Parquet) is on `/mnt/aux`, which has 761 GB free.